## Feature Engineering

In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [64]:
df = pd.read_parquet('../data/processed/paysim_clean.parquet')

In [65]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,dest_is_merchant
0,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0,0
1,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0,0
2,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0,0
3,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0,0
4,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0,0


In [66]:
df.shape

(2770409, 12)

In [67]:
y = df['isFraud']

In [68]:
cols_to_drop = [
    'nameOrig',
    'nameDest',
    'isFlaggedFraud',
    'isFraud',
    'newbalanceOrig',
    'newbalanceDest'
    
]

In [69]:
df['log_amount'] = np.log1p(df['amount'])

In [70]:
print(df[['amount', 'log_amount']].describe())

             amount    log_amount
count  2.770409e+06  2.770409e+06
mean   3.175361e+05  1.192780e+01
std    8.877897e+05  1.232817e+00
min    0.000000e+00  0.000000e+00
25%    8.297354e+04  1.132629e+01
50%    1.712609e+05  1.205095e+01
75%    3.067912e+05  1.263393e+01
max    9.244552e+07  1.834213e+01


In [71]:
df['log_amount'].dtypes

dtype('float64')

In [72]:
df['log_amount'].isnull().sum()

0

In [73]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,dest_is_merchant,log_amount
0,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0,0,5.204007
1,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0,0,5.204007
2,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0,0,12.342066
3,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0,0,12.279840
4,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0,0,12.649754


In [74]:
df['amount_to_oldbalance_ratio'] =  df['amount'] / (df['oldbalanceOrg'] + 1)
    


In [75]:
print(df[['amount', 'oldbalanceOrg', 'amount_to_oldbalance_ratio']].head())

      amount  oldbalanceOrg  amount_to_oldbalance_ratio
0     181.00          181.0                    0.994505
1     181.00          181.0                    0.994505
2  229133.94        15325.0                   14.950668
3  215310.30          705.0                  304.972096
4  311685.89        10835.0                   28.763925


In [76]:
print(df.iloc[[2, 3, 4]]['isFraud'])

2    0
3    0
4    0
Name: isFraud, dtype: int64


In [77]:
print(
    "Infinite values:",
    np.isinf(df['amount_to_oldbalance_ratio']).sum()
)

Infinite values: 0


In [78]:
print(
    "Missing values:",
    df['amount_to_oldbalance_ratio'].isna().sum()
)

Missing values: 0


In [79]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,dest_is_merchant,log_amount,amount_to_oldbalance_ratio
0,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0,0,5.204007,0.994505
1,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0,0,5.204007,0.994505
2,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0,0,12.342066,14.950668
3,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0,0,12.279840,304.972096
4,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0,0,12.649754,28.763925


In [80]:
df['sender_near_empty'] = np.where(
    (df['oldbalanceOrg'] > 0) &
    ((df['amount'] / df['oldbalanceOrg']) >= 0.95),
    1,
    0
)

In [81]:
df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,dest_is_merchant,log_amount,amount_to_oldbalance_ratio,sender_near_empty
0,1,TRANSFER,181.00,C1305486145,181.00,0.0,C553264065,0.0,0.00,1,0,0,5.204007,0.994505,1
1,1,CASH_OUT,181.00,C840083671,181.00,0.0,C38997010,21182.0,0.00,1,0,0,5.204007,0.994505,1
2,1,CASH_OUT,229133.94,C905080434,15325.00,0.0,C476402209,5083.0,51513.44,0,0,0,12.342066,14.950668,1
3,1,TRANSFER,215310.30,C1670993182,705.00,0.0,C1100439041,22425.0,0.00,0,0,0,12.279840,304.972096,1
4,1,TRANSFER,311685.89,C1984094095,10835.00,0.0,C932583850,6267.0,2719172.89,0,0,0,12.649754,28.763925,1
5,1,CASH_OUT,110414.71,C768216420,26845.41,0.0,C1509514333,288800.0,2415.16,0,0,0,11.612008,4.112830,1
6,1,CASH_OUT,56953.90,C1570470538,1942.02,0.0,C824009085,70253.0,64106.18,0,0,0,10.950015,29.312050,1
7,1,CASH_OUT,5346.89,C512549200,0.00,0.0,C248609774,652637.0,6453430.91,0,0,0,8.584457,5346.890000,0
8,1,CASH_OUT,23261.30,C2072313080,20411.53,0.0,C2001112025,25742.0,0.00,0,0,0,10.054589,1.139560,1
9,1,TRANSFER,62610.80,C1976401987,79114.00,16503.2,C1937962514,517.0,8383.29,0,0,0,11.044709,0.791390,0


In [103]:
df['sender_near_empty'].value_counts()

sender_near_empty
0    1572057
1    1198352
Name: count, dtype: int64

In [82]:
print(
    df[['amount', 'oldbalanceOrg', 
        'amount_to_oldbalance_ratio', 
        'sender_near_empty']].head(10)
)

      amount  oldbalanceOrg  amount_to_oldbalance_ratio  sender_near_empty
0     181.00         181.00                    0.994505                  1
1     181.00         181.00                    0.994505                  1
2  229133.94       15325.00                   14.950668                  1
3  215310.30         705.00                  304.972096                  1
4  311685.89       10835.00                   28.763925                  1
5  110414.71       26845.41                    4.112830                  1
6   56953.90        1942.02                   29.312050                  1
7    5346.89           0.00                 5346.890000                  0
8   23261.30       20411.53                    1.139560                  1
9   62610.80       79114.00                    0.791390                  0


In [83]:
df['hour'] = df['step'] % 24
print(df[['step', 'hour']].head(10))

   step  hour
0     1     1
1     1     1
2     1     1
3     1     1
4     1     1
5     1     1
6     1     1
7     1     1
8     1     1
9     1     1


In [84]:
print(df['hour'].min())
print(df['hour'].max())

0
23


In [104]:
df['is_night'] = df['hour'].between(0,5).astype('int8')

In [109]:
print(df[['step', 'hour', 'is_night']].head(10))
df['hour'].value_counts()
df['is_night'].value_counts()

     step  hour  is_night
0       1     1         1
438     1     1         1
439     1     1         1
440     1     1         1
441     1     1         1
442     1     1         1
443     1     1         1
444     1     1         1
437     1     1         1
445     1     1         1


is_night
0    2742344
1      28065
Name: count, dtype: int64

In [110]:
## Encode the transaction type
df['is_transfer'] = (df['type'] == 'TRANSFER').astype('int8')


In [87]:
df['type_encoded'] = df['type'].map({
    'CASH_OUT': 0,
    'TRANSFER': 1
})

In [88]:
print(df[['type', 'type_encoded']].head(10))

       type type_encoded
0  TRANSFER            1
1  CASH_OUT            0
2  CASH_OUT            0
3  TRANSFER            1
4  TRANSFER            1
5  CASH_OUT            0
6  CASH_OUT            0
7  CASH_OUT            0
8  CASH_OUT            0
9  TRANSFER            1


In [89]:
print(df['type_encoded'].value_counts())

type_encoded
0    2237500
1     532909
Name: count, dtype: int64


In [90]:
print(df['type_encoded'].isna().sum())

0


In [91]:
print(df['dest_is_merchant'].info())

<class 'pandas.core.series.Series'>
RangeIndex: 2770409 entries, 0 to 2770408
Series name: dest_is_merchant
Non-Null Count    Dtype
--------------    -----
2770409 non-null  int64
dtypes: int64(1)
memory usage: 21.1 MB
None


In [92]:
print("Missing values:", df['dest_is_merchant'].isna().sum())
print("Unique values:", df['dest_is_merchant'].unique())


Missing values: 0


Unique values: [0]


In [ ]:
# Sort transactions by transaction time
df = df.sort_values('step')


# Count only previous transactions received by each destination
df['dest_prior_txn_count'] = (
    df.groupby('nameDest').cumcount()
)

In [111]:
df['dest_prior_txn_count'].describe()

count    2.770409e+06
mean     5.291217e+00
std      6.011013e+00
min      0.000000e+00
25%      1.000000e+00
50%      3.000000e+00
75%      8.000000e+00
max      7.400000e+01
Name: dest_prior_txn_count, dtype: float64

In [94]:
print(
    df[
        df['nameDest'].duplicated(keep=False)
    ][[
        'step',
        'nameDest',
        'dest_prior_txn_count'
    ]].sort_values(['nameDest', 'step']).head(20)
)

         step     nameDest  dest_prior_txn_count
2200768   354  C1000004082                     0
2278389   370  C1000004082                     1
2328749   374  C1000004082                     2
501457    131  C1000004940                     0
775902    161  C1000004940                     1
886773    180  C1000004940                     2
1136657   207  C1000004940                     3
1205109   212  C1000004940                     4
1754642   299  C1000004940                     5
1814986   303  C1000004940                     6
1856491   306  C1000004940                     7
2290102   371  C1000004940                     8
2357084   377  C1000004940                     9
853929    177  C1000013769                     0
1229894   225  C1000013769                     1
1284697   229  C1000013769                     2
1416446   250  C1000013769                     3
1962009   324  C1000013769                     4
2034010   330  C1000013769                     5
2229541   355  C1000

In [95]:
 print(df['dest_prior_txn_count'].isna().sum())

0


In [ ]:
first_txn = df.groupby('nameDest')['dest_prior_txn_count'].min()

print(first_txn.value_counts().head())

dest_prior_txn_count
0    509565
Name: count, dtype: int64


LEAKAGE CHECK
-------------

newbalanceOrig and newbalanceDest contain account balances after the transaction.
Although these variables and features derived from them may appear highly predictive
of fraud, they are not valid live features for a production-style model that makes
the fraud decision at authorization time.

Any feature directly derived from these post-transaction columns is also considered
leaky. Examples include orig_balance_change, dest_balance_change, and
orig_balance_error.

These columns may remain in the working DataFrame for EDA and model comparison,
but they must NOT be included in the final production feature matrix.

In [ ]:
#df = df.sort_values(['nameOrig', 'step'], kind='stable')

#df['sender_time_since_prev'] = (
#    df['step'] - df['sender_prev_step']
#).fillna(-1)

In [116]:
df['sender_zero_balance'] = (df['oldbalanceOrg'] == 0).astype('int8')

In [117]:
print(df.groupby('sender_zero_balance')['isFraud'].agg(rows='size',fraud_rate='mean'))
print(df.groupby('sender_near_empty')['isFraud'].agg(rows='size',fraud_rate='mean'))

                        rows  fraud_rate
sender_zero_balance                     
0                    1461827    0.005590
1                    1308582    0.000031
                      rows  fraud_rate
sender_near_empty                     
0                  1572057    0.000118
1                  1198352    0.006699


In [118]:
STRICT = ["log_amount", "is_transfer", "hour", "is_night", "dest_prior_txn_count"]

STANDARD = STRICT + ["amount_to_oldbalance_ratio", "sender_near_empty", "sender_zero_balance"]

FEATURE_SETS = {"STRICT": STRICT, "STANDARD": STANDARD}

FORBIDDEN = {"isFraud", "step", "nameOrig", "nameDest", "isFlaggedFraud",
             "newbalanceOrig", "newbalanceDest", "type", "amount", "oldbalanceOrg", "oldbalanceDest"}
for name, cols in FEATURE_SETS.items():
    assert not (FORBIDDEN & set(cols)), f"{name} contains a forbidden column: {FORBIDDEN & set(cols)}"
    assert (df[cols].nunique() > 1).all(), f"{name} contains a constant column"
    print(f"{name:<9}({len(cols)}):", cols)

STRICT   (5): ['log_amount', 'is_transfer', 'hour', 'is_night', 'dest_prior_txn_count']
STANDARD (8): ['log_amount', 'is_transfer', 'hour', 'is_night', 'dest_prior_txn_count', 'amount_to_oldbalance_ratio', 'sender_near_empty', 'sender_zero_balance']


## Validation

In [119]:
ALL_FEATURES = sorted(set(STANDARD))
out_cols = ALL_FEATURES + ["isFraud", "step"]
features_df = df[out_cols].copy()

checks = pd.DataFrame({
    "dtype":    features_df[ALL_FEATURES].dtypes.astype(str),
    "missing":  features_df[ALL_FEATURES].isna().sum(),
    "infinite": np.isinf(features_df[ALL_FEATURES]).sum(),
    "n_unique": features_df[ALL_FEATURES].nunique(),
})
print("rows:", f"{len(features_df):,}", "| fraud rate:", f"{features_df['isFraud'].mean()*100:.4f}%")
checks

rows: 2,770,409 | fraud rate: 0.2965%


,dtype,missing,infinite,n_unique
amount_to_oldbalance_ratio,float64,0,0,2747633
dest_prior_txn_count,int64,0,0,75
hour,int64,0,0,24
is_night,int8,0,0,2
is_transfer,int8,0,0,2
log_amount,float64,0,0,2685805
sender_near_empty,int64,0,0,2
sender_zero_balance,int8,0,0,2


In [121]:

import json
from pathlib import Path

# Validate features
assert checks["missing"].sum() == 0, "features contain missing values"
assert checks["infinite"].sum() == 0, "features contain infinite values"
assert features_df["step"].is_monotonic_increasing, \
    "data must be in time order"

# Define output paths
features_path = Path("../data/processed/paysim_features.parquet")
sets_path = Path("../data/processed/feature_sets.json")

# Create directory if it doesn't exist
features_path.parent.mkdir(parents=True, exist_ok=True)

# Save feature data
features_df.to_parquet(features_path, index=False)

# Save feature-set definitions
sets_path.write_text(
    json.dumps(FEATURE_SETS, indent=2),
    encoding="utf-8"
)

print("Saved:", features_path, features_df.shape)
print("Saved:", sets_path)


Saved: ../data/processed/paysim_features.parquet (2770409, 10)
Saved: ../data/processed/feature_sets.json
